# 06 -- Expected Loss (EL = PD x LGD x EAD)

**What this notebook does (plain English):** Brings the three pieces together.
**Expected Loss** is the average loss a lender should budget for:

> **Expected Loss = chance of default (PD) x loss if it defaults (LGD) x amount
> owed (EAD)**

We score every loan, total it into a portfolio number, and sort loans into the
accounting **IFRS 9 / AASB 9 stages** (1 = healthy, 2 = deteriorating, 3 =
defaulted). We also walk through the full sum for one example loan.

**Headline result:** a single portfolio Expected Loss figure, dominated by the
Stage 3 (already-defaulted) loans and by the crisis vintages.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and build the three components for EVERY loan.
import pandas as pd
import numpy as np
from src import models
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet').copy()

In [3]:
# One-year PD for every loan (logistic model fit on the whole book). pd_hat is a
# 12-month PD (PD-1/PD-2) -- exactly the IFRS 9 Stage 1 (12-month ECL) input.
from src import definitions as d
pd_model, pd_cols = models.fit_pd(base)
base['pd_hat'] = models.predict_pd(pd_model, pd_cols, base)
# PD-6: apply the 5 bps regulatory PD floor to the per-loan PD used in EL.
base['pd_hat'] = d.apply_pd_floor(base['pd_hat'], floor=0.0005)

In [4]:
# LGD for every loan (two-stage model trained on disposed defaults).
disposed = base[base['disposed'] & base['lgd'].notna()]
lgd_model = models.TwoStageLGD().fit(disposed)
base['lgd_hat'] = lgd_model.predict(base)

In [5]:
# EAD for every loan: balance at default if it defaulted, else the original
# loan amount as the exposure proxy for a still-performing loan.
base['ead_loan'] = np.where(base['ever_default'], base['ead'], base['original_upb'])
# Expected loss per loan = PD x LGD x EAD.
base['expected_loss'] = base['pd_hat'] * base['lgd_hat'] * base['ead_loan']

In [6]:
# IFRS 9 / AASB 9 staging: 3 = defaulted (credit-impaired), 2 = significant
# increase in risk (ever 60+ days late but not defaulted), 1 = performing.
stage2 = (~base['ever_default']) & (base['max_delinq_status'].fillna(0) >= 2)
base['ifrs9_stage'] = np.where(base['ever_default'], 3, np.where(stage2, 2, 1))
# pd_hat is now a genuine 12-MONTH PD, which is exactly the Stage 1 (12-month ECL)
# input -- so Stage 1 reported EL is the 12-month EL directly (no ad-hoc 0.25 factor
# any more). Stages 2 & 3 need LIFETIME ECL; with only a one-year PD modelled here we
# scale by a transparent multi-year horizon factor as a lifetime proxy (a production
# model would estimate lifetime PD directly).
LIFETIME_HORIZON = 4
base['el_reported'] = np.where(base['ifrs9_stage'] == 1, base['expected_loss'],
                               base['expected_loss'] * LIFETIME_HORIZON)

In [7]:
# Portfolio Expected Loss summary by IFRS 9 stage (the saved result).
el_summary = base.groupby('ifrs9_stage').agg(
    loans=('loan_sequence_number', 'size'),
    avg_pd=('pd_hat', 'mean'),
    avg_lgd=('lgd_hat', 'mean'),
    total_ead=('ead_loan', 'sum'),
    lifetime_expected_loss=('expected_loss', 'sum'),
    reported_expected_loss=('el_reported', 'sum'),
).reset_index().round(2)
save_csv(el_summary, 'output/06_expected_loss.csv')
el_summary

,ifrs9_stage,loans,avg_pd,avg_lgd,total_ead,lifetime_expected_loss,reported_expected_loss
0,1,132177,0.01,0.44,2.703702e+10,74618768.45,7.461877e+07
1,2,6067,0.02,0.50,1.140920e+09,9567875.32,3.827150e+07
2,3,11756,0.02,0.53,2.247190e+09,29154300.09,1.166172e+08


### Downturn-LGD variant of Expected Loss (P2-3)

Notebook 04 showed loss severity is strongly **cyclical** (~25% calm vs ~57% crisis).
APS 113 Att D LGD paras 4-5 say that where severity is cyclical, the LGD *estimate*
must reflect **downturn** conditions, not the through-the-cycle average. So alongside
the baseline EL we compute a **downturn-LGD variant**, lifting every loan's LGD to at
least the crisis-regime realised severity. This is the conservative figure the
framework expects a capital/EL report to show.

In [8]:
# P2-3: downturn-LGD variant of EL. Lift each loan's LGD to >= the crisis-regime
# realised severity (the observed downturn LGD), then recompute Expected Loss.
downturn_lgd = float(base.loc[base['disposed'] & base['vintage_year'].isin([2007, 2008]), 'lgd'].mean())
base['lgd_downturn'] = np.maximum(base['lgd_hat'], downturn_lgd)
base['expected_loss_downturn'] = base['pd_hat'] * base['lgd_downturn'] * base['ead_loan']
el_variant = pd.DataFrame([
    {'view': 'through-the-cycle (baseline)', 'lgd_basis': 'modelled lgd_hat',
     'total_expected_loss': round(float(base['expected_loss'].sum()), 0)},
    {'view': 'downturn LGD (APS 113 Att D LGD 4-5)', 'lgd_basis': f'max(lgd_hat, {downturn_lgd:.3f})',
     'total_expected_loss': round(float(base['expected_loss_downturn'].sum()), 0)},
])
el_variant['uplift_x'] = (el_variant['total_expected_loss'] /
                          el_variant['total_expected_loss'].iloc[0]).round(2)
save_csv(el_variant, 'output/06_el_downturn_variant.csv')
el_variant

,view,lgd_basis,total_expected_loss,uplift_x
0,through-the-cycle (baseline),modelled lgd_hat,113340944.0,1.00
1,downturn LGD (APS 113 Att D LGD 4-5),"max(lgd_hat, 0.567)",129463750.0,1.14


### Best estimate of EL for already-defaulted (Stage 3) loans (P2-4)

APS 113 Att D para 11 / Part 4.3: for loans **already in default** (Stage 3), you must
form a **best estimate of expected loss for that loan given current conditions** --
mechanically applying the model's average LGD is "not acceptable". We replace the
mechanical PD x LGD with: the loan's **realised** LGD where its workout is materially
complete (disposed), otherwise the **segment downturn LGD**; since the loan is already
in default its PD is 1, so EL = best-estimate LGD x EAD.

In [9]:
# P2-4: best-estimate EL for Stage 3 (already-defaulted) loans.
stage3 = base['ifrs9_stage'] == 3
best_lgd = np.where(base['disposed'] & base['lgd'].notna(), base['lgd'], downturn_lgd)
base['el_stage3_bestestimate'] = np.where(stage3, best_lgd * base['ead_loan'], np.nan)
s3 = pd.DataFrame([{
    'stage3_loans': int(stage3.sum()),
    'el_mechanical_pd_x_lgd': round(float(base.loc[stage3, 'expected_loss'].sum()), 0),
    'el_best_estimate': round(float(np.nansum(base['el_stage3_bestestimate'])), 0),
}])
s3['ratio_best_vs_mechanical'] = round(s3['el_best_estimate'] / s3['el_mechanical_pd_x_lgd'], 2)
save_csv(s3, 'output/06_stage3_best_estimate.csv')
s3

,stage3_loans,el_mechanical_pd_x_lgd,el_best_estimate,ratio_best_vs_mechanical
0,11756,29154300.0,1.223179e+09,41.96


**Reading the Stage 3 table (P2-4).** The mechanical column applies the model
PD x LGD even to loans that have *already* defaulted (so its PD < 1 understates the
loss); the best-estimate column uses each defaulted loan's realised loss where the
workout is complete and the downturn LGD otherwise, with PD = 1. The best estimate is
materially larger -- which is the point: a defaulted loan's expected loss should be
built from its own resolution, not a portfolio-average model output.

In [10]:
# Worked example: show PD x LGD x EAD = EL for a single representative loan.
ex = base.sort_values('expected_loss', ascending=False).iloc[100]
print('Worked example loan:', ex['loan_sequence_number'])
print(f"  PD  (chance of default) = {ex['pd_hat']:.3f}")
print(f"  LGD (loss if default)   = {ex['lgd_hat']:.3f}")
print(f"  EAD (amount owed)       = ${ex['ead_loan']:,.0f}")
print(f"  Expected Loss = {ex['pd_hat']:.3f} x {ex['lgd_hat']:.3f} x ${ex['ead_loan']:,.0f} = ${ex['expected_loss']:,.0f}")

Worked example loan: F07Q30168418
  PD  (chance of default) = 0.134
  LGD (loss if default)   = 0.456
  EAD (amount owed)       = $353,954
  Expected Loss = 0.134 x 0.456 x $353,954 = $21,637


**Reading the table:** Stage 3 holds the already-defaulted loans and
carries most of the loss; Stage 1 is the large healthy book on a 12-month view.
The worked example shows the headline equation end-to-end for one loan.